# CS224N · Lecture 18 — Deployment and Efficiency: Hardware-Aware Algorithms for Sequence Modeling

**Slides:** [cs224n-2024-lecture18-deployment-and-efficiency.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-2024-lecture18-deployment-and-efficiency.pdf) (Tri Dao; Mamba slides credit Albert Gu)  
**Related notebooks:** L8 (attention, KV cache, efficient-attention variants), L12 (mixed precision, memory), L16 §6 (state space models as convolutions).

**Scale brings quality and capabilities** (from 100 million parameters in 2018 to 500 billion in 2022). The slides' example: asked to explain *"I tried 10000 random restarts of my neural network, but I was accused of overfitting. I guess no good seed goes unpunished."*, a 1.3B model misses the pun; a 175B model explains the play on "no good deed goes unpunished." **The core challenge with scale is efficiency**, and efficiency unlocks new capabilities (e.g., long context), and eases training, deployment, and research.

**Main idea: hardware-aware algorithms.** Understand both the fundamental algorithms and the hardware (block-oriented devices, an asymmetric memory hierarchy):
* **IO-awareness:** reducing reads/writes to GPU memory yields significant speedups → **FlashAttention**: fast, memory-efficient attention **with no approximation**.
* **State-space expansion:** expand recurrent states only in fast on-chip memory, to avoid memory cost → **Mamba**: a selective state space model that matches Transformers on language modeling, with fast inference and up to 1M context.

| § | Topic | You will implement / verify |
|---|---|---|
| 1 | Long sequences; why attention is bottlenecked by **memory reads/writes** | Arithmetic intensity and the roofline; HBM traffic of standard attention |
| 2 | **FlashAttention**: tiling with softmax rescaling, recomputation | Blockwise forward (exact), **recomputation backward** (exact), IO counts |
| 3 | FlashAttention-2/3, **Flash-Decoding** | Split-KV decoding with a log-sum-exp reduction (exact) |
| 4 | State space models: continuous, recurrent, convolutional views | Zero-order-hold discretization, checked against the ODE |
| 5 | **Selection** (Mamba): why LTI models fail, the selective scan | LTI vs. selective SSM on a selective task; the **parallel scan** |
| 6 | Results and implications for foundation models | — |
| 7 | 🔎 Deployment essentials: KV-cache memory, **quantization**, **speculative decoding**, batching | int8/int4 quantization error; speculative sampling, verified exact |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math, collections, time
from scipy.linalg import expm

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True); e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)

---
## 1. Motivation: modeling long sequences, and where the time actually goes

**Long sequences** are everywhere: NLP (books, plays, codebases), computer vision (higher resolution → more robust insight), time series, audio, video, and medical imaging (sequences of millions of steps). **Efficiency is the bottleneck for modeling long sequences with attention**: increasing the context length slows down (or stops) training.

**Attention** (L8): $O = \text{softmax}(QK^\top)V$ with $Q, K, V\in\mathbb{R}^{N\times d}$, $S = QK^\top\in\mathbb{R}^{N\times N}$, $A = \text{softmax}(S)$ row-wise. Typical $N$: 1K–8K; head dimension $d$: 64–128. **Attention scales quadratically in $N$.** Approximate attention methods (sparse, low-rank; L8 §9, Tay et al.'s Long Range Arena) trade quality for speed with fewer FLOPs, but often don't deliver wall-clock speedups. **Is there a fast, memory-efficient, and exact attention algorithm?**

### 1.1 The key observation: attention is bottlenecked by memory reads/writes
**The biggest cost is in moving the bits!** A GPU has an asymmetric **memory hierarchy** (numbers for an A100, from the FlashAttention paper):

| Memory | Size | Bandwidth |
|---|---|---|
| GPU **SRAM** (on-chip, per streaming multiprocessor) | ~20 MB total | ~19 TB/s |
| GPU **HBM** (high-bandwidth memory, "GPU memory") | 40–80 GB | ~1.5–2.0 TB/s |
| CPU DRAM | > 1 TB | ~12.8 GB/s |

The GPU compute model: (1) inputs start out in HBM; (2) data is moved to the compute units and SRAM for computation; (3) the output is written back to HBM. The standard attention implementation **materializes $S$ and $A$ ($N\times N$) in HBM**, with repeated reads and writes.

**Arithmetic intensity** (FLOPs per byte moved) decides whether an operation is **compute-bound** or **memory-bound**. An A100 does ~312 TFLOP/s (BF16 tensor cores) but moves only ~1.5–2 TB/s, so an operation needs roughly **150–200 FLOPs per byte** to keep the compute units busy (the "ridge point" of the roofline model). Big matrix multiplies are above it; softmax, masking, dropout, and other elementwise operations are far below it.

In [ ]:
PEAK_FLOPS, HBM_BW = 312e12, 1.555e12             # A100 40GB: BF16 tensor-core peak, HBM bandwidth (bytes/s)
ridge = PEAK_FLOPS / HBM_BW
def roofline_time(flops, bytes_moved):
    return max(flops / PEAK_FLOPS, bytes_moved / HBM_BW), ("compute-bound" if flops / bytes_moved > ridge else "memory-bound")

N, d, b = 4096, 64, 2                              # sequence length, head dim, bytes per element (fp16)
ops = {
    "S = Q K^T     (matmul)": (2 * N * N * d, b * (2 * N * d + N * N)),
    "A = softmax(S) (elementwise)": (5 * N * N, b * (2 * N * N)),
    "O = A V       (matmul)": (2 * N * N * d, b * (N * N + 2 * N * d)),
}
print(f"ridge point: {ridge:.0f} FLOPs/byte")
total = 0
for name, (fl, by) in ops.items():
    t, kind = roofline_time(fl, by); total += t
    print(f"{name:30s} arithmetic intensity {fl / by:6.1f} FLOPs/byte -> {kind:13s} ~{t * 1e6:6.1f} µs")
print(f"standard attention, one head: ~{total * 1e6:.0f} µs, dominated by moving the N x N matrices")

Even the matmuls have modest intensity here (with $d = 64$, each element of $S$ costs only $2d$ FLOPs to produce), and the softmax is extremely memory-bound. Most of the time goes to **writing and re-reading the $N\times N$ matrices** in HBM, not to arithmetic. (Real kernels also fuse masking, scaling, and dropout; the point is the same.) *Blog: Horace He, "Making Deep Learning Go Brrrr From First Principles."*

---
## 2. FlashAttention (Dao, Fu, Ermon, Rudra & Ré, NeurIPS 2022)

**How to reduce HBM reads/writes: compute by blocks.**
1. **Tiling:** restructure the algorithm to load **block by block** from HBM to SRAM to compute attention.
2. **Recomputation:** don't store the attention matrix from the forward pass; **recompute it in the backward pass**.

**Challenges:** (1) compute the softmax normalization **without access to the full input** (a row's softmax needs all of that row's scores); (2) do the backward pass **without the large attention matrix** from the forward pass.

### 2.1 Tiling with softmax rescaling
Split $K, V$ into blocks $K^{(1)}, K^{(2)}, \dots$ For a block of queries, keep a running **max** $m$, running **normalizer** $\ell = \sum_i e^{S_i - m}$, and running **unnormalized output**. When a new block arrives with scores $S^{(j)}$:
$$m_{\text{new}} = \max(m, \text{rowmax}(S^{(j)})),\quad \ell_{\text{new}} = e^{m - m_{\text{new}}}\ell + \text{rowsum}\big(e^{S^{(j)} - m_{\text{new}}}\big),\quad O_{\text{new}} = \frac{e^{m - m_{\text{new}}}\ell\,O + e^{S^{(j)} - m_{\text{new}}}V^{(j)}}{\ell_{\text{new}}}$$
The first attempt in the slides (just summing local results) has the **wrong denominator**; **rescaling** the previous partial output by the ratio of old and new normalizers fixes it. **Tiling + rescaling allows local computation in SRAM, without writing $S$ or $A$ to HBM, and gets the right answer!** We also save the per-row **log-sum-exp** $L = m + \log\ell$ (size $N$) for the backward pass.

In [ ]:
class HBM:
    # Counts elements read from / written to 'HBM' (our simulated slow memory).
    def __init__(self):
        self.reads = self.writes = 0
    def read(self, x):
        self.reads += x.size; return x
    def write(self, x):
        self.writes += x.size; return x

def standard_attention(Q, K, V, hbm):
    S = hbm.write(hbm.read(Q) @ hbm.read(K).T / np.sqrt(Q.shape[1]))     # S materialized in HBM
    P = hbm.write(softmax(hbm.read(S), 1))                               # P materialized in HBM
    O = hbm.write(hbm.read(P) @ hbm.read(V))
    return O, P

def flash_attention_forward(Q, K, V, Br, Bc, hbm):
    N, d = Q.shape; scale = 1 / np.sqrt(d)
    O = np.zeros((N, d)); L = np.zeros(N)
    for i in range(0, N, Br):                                            # outer loop: blocks of queries
        Qi = hbm.read(Q[i:i + Br])
        m = np.full(len(Qi), -np.inf); l = np.zeros(len(Qi)); Oi = np.zeros((len(Qi), d))   # running stats (in SRAM)
        for j in range(0, N, Bc):                                        # inner loop: blocks of keys/values
            Kj, Vj = hbm.read(K[j:j + Bc]), hbm.read(V[j:j + Bc])
            Sij = Qi @ Kj.T * scale                                      # computed in SRAM, never written to HBM
            m_new = np.maximum(m, Sij.max(1))
            Pij = np.exp(Sij - m_new[:, None])
            l_new = np.exp(m - m_new) * l + Pij.sum(1)
            Oi = ((np.exp(m - m_new) * l)[:, None] * Oi + Pij @ Vj) / l_new[:, None]       # rescale + accumulate
            m, l = m_new, l_new
        O[i:i + Br] = hbm.write(Oi); L[i:i + Br] = hbm.write(m + np.log(l))   # save only O and the log-sum-exp
    return O, L

N, d = 512, 64
Q, K, V = (rng.normal(size=(N, d)) for _ in range(3))
h_std, h_flash = HBM(), HBM()
O_std, P_std = standard_attention(Q, K, V, h_std)
O_fl, L_fl = flash_attention_forward(Q, K, V, Br=64, Bc=64, hbm=h_flash)
print(f"max |FlashAttention - standard| = {np.abs(O_fl - O_std).max():.1e}   (exact, up to float rounding)")
for name, h in [("standard", h_std), ("FlashAttention", h_flash)]:
    print(f"{name:15s} HBM traffic: {h.reads + h.writes:>10,} elements")

In this simple simulation (N = 512, blocks of 64), FlashAttention moves about **half** as much data; it re-reads the $K, V$ blocks once per query block, so its traffic is $O(N^2d/B_r)$ elements instead of $O(N^2)$ for materializing $S$ and $P$, and the gap widens with longer sequences and bigger blocks. The paper's analysis: standard attention needs $\Theta(Nd + N^2)$ HBM accesses, FlashAttention $\Theta(N^2d^2M^{-1})$ for SRAM size $M$, which is many times fewer for typical $d$ (64–128) and $M$ (~100 KB). The benefit grows with the block size (bigger SRAM), and the output is identical.

### 2.2 Recomputation in the backward pass
By storing the softmax normalization from the forward pass (**size $N$**), we can quickly **recompute** attention blocks in the backward pass from $Q, K$ in SRAM. With $P = \exp(S - L)$ recomputed per block and $D_i = \text{rowsum}(dO_i\odot O_i)$:
$$dV \mathrel{+}= P^\top dO,\qquad dP = dO\,V^\top,\qquad dS = P\odot(dP - D),\qquad dQ \mathrel{+}= dS\,K/\sqrt d,\qquad dK \mathrel{+}= dS^\top Q/\sqrt d$$
(the softmax backward of L8, written blockwise). **FlashAttention speeds up the backward pass even with increased FLOPs.** From the paper, for one GPT-2-medium-size attention: standard **66.6 GFLOPs, 40.3 GB of HBM reads/writes, 41.7 ms**; FlashAttention **75.2 GFLOPs (↑13%), 4.4 GB (↓9×), 7.3 ms (↓6×)**.

In [ ]:
def standard_attention_backward(Q, K, V, P, O, dO):
    scale = 1 / np.sqrt(Q.shape[1])
    dV = P.T @ dO; dP = dO @ V.T
    dS = P * (dP - (dP * P).sum(1, keepdims=True))                     # softmax backward
    return dS @ K * scale, dS.T @ Q * scale, dV

def flash_attention_backward(Q, K, V, O, L, dO, Br, Bc):
    N, d = Q.shape; scale = 1 / np.sqrt(d)
    dQ, dK, dV = np.zeros_like(Q), np.zeros_like(K), np.zeros_like(V)
    D = (dO * O).sum(1)                                                 # = rowsum(dP * P), computable from O and dO
    for j in range(0, N, Bc):
        Kj, Vj = K[j:j + Bc], V[j:j + Bc]
        for i in range(0, N, Br):
            Qi, dOi = Q[i:i + Br], dO[i:i + Br]
            Pij = np.exp(Qi @ Kj.T * scale - L[i:i + Br, None])          # RECOMPUTE the attention block from the stored L
            dV[j:j + Bc] += Pij.T @ dOi
            dSij = Pij * (dOi @ Vj.T - D[i:i + Br, None])
            dQ[i:i + Br] += dSij @ Kj * scale; dK[j:j + Bc] += dSij.T @ Qi * scale
    return dQ, dK, dV

dO = rng.normal(size=(N, d))
g_std = standard_attention_backward(Q, K, V, P_std, O_std, dO)
g_fl = flash_attention_backward(Q, K, V, O_fl, L_fl, dO, 64, 64)
for name, a_, b_ in zip(["dQ", "dK", "dV"], g_std, g_fl):
    print(f"{name}: max |flash - standard| = {np.abs(a_ - b_).max():.1e}")
# numerical check of one entry of the standard gradient, to be sure both are right
eps = 1e-6; Q2 = Q.copy(); Q2[3, 5] += eps
num = ((standard_attention(Q2, K, V, HBM())[0] - O_std) * dO).sum() / eps
print(f"numerical dQ[3,5] = {num:.6f}   analytic = {g_std[0][3, 5]:.6f}")
print(f"memory kept between forward and backward: standard {N * N:,} (the matrix P) vs flash {N:,} (L)")

**Results.** FlashAttention gives a **2–4× speedup with no approximation**, and a **10–20× memory reduction** (memory linear in sequence length). For GPT-3-style training (Megatron-LM baseline): **2× faster training, 4× longer context, and better models**. On the Pile: GPT-1.3B with 2K context reaches validation perplexity 5.45, with 8K context **5.24**; GPT-2.7B, 2K: 5.02, 8K: **4.87**. Long context became affordable, and it helps.

---
## 3. Follow-ups: FlashAttention-2, FlashAttention on H100, and Flash-Decoding

* **FlashAttention-2** (Dao 2023): **reduce non-matmul FLOPs** (GPUs are much faster at matmuls than at other operations, so every non-matmul FLOP is expensive); **parallelize over the sequence-length dimension** to improve GPU occupancy; **better work partitioning between warps** to reduce communication. Upshot: **2× faster** wall-clock; train models with 2× the context length for the same cost.
* **Optimizing for H100** (Bikshandi & Shah): new hardware features, the **wgmma** instruction (higher matmul throughput), **TMA** (faster loading between global and shared memory), and **FP8** (lower precision, higher throughput). Upshot: 1.2–2.5× speedup. (FlashAttention-3 later combined these.)
* **Flash-Decoding** (Dao, Haziza, Massa & Sizov): during **decoding**, there's only **one query** per step, so the previous approaches, which parallelize across blocks of queries, batch size, and heads only, **don't occupy the entire GPU**, and loading the KV cache is slow. Flash-Decoding **parallelizes the KV cache over the sequence-length dimension**: split the keys/values into chunks, compute attention against each chunk in parallel (keeping each chunk's max and normalizer), then combine with a separate **reduction step**. Upshot: **2–8× faster end-to-end generation** on CodeLlama 34B with 32K–100K context. *The decoding IO bottleneck is all about loading the KV cache as fast as possible.*

In [ ]:
def flash_decode(q, K, V, n_chunks):
    # One query against a long KV cache, split into chunks processed independently, then combined.
    partial = []
    for Kc, Vc in zip(np.array_split(K, n_chunks), np.array_split(V, n_chunks)):   # each chunk on a different SM
        s = Kc @ q / np.sqrt(len(q)); m = s.max(); e = np.exp(s - m)
        partial.append((m, e.sum(), e @ Vc / e.sum()))                             # (max, normalizer, local output)
    m_all = max(m for m, _, _ in partial)                                          # reduction step
    w = np.array([l * np.exp(m - m_all) for m, l, _ in partial])
    return sum(wi * o for wi, (_, _, o) in zip(w, partial)) / w.sum()

T_ctx, d = 32_768, 128
Kc, Vc = rng.normal(size=(T_ctx, d)), rng.normal(size=(T_ctx, d)); q = rng.normal(size=d)
ref = softmax(Kc @ q / np.sqrt(d)) @ Vc
for n_chunks in [1, 8, 64]:
    print(f"{n_chunks:3d} chunks: max |flash-decoding - exact| = {np.abs(flash_decode(q, Kc, Vc, n_chunks) - ref).max():.1e}")

**Summary of FlashAttention:** a fast and memory-efficient algorithm for **exact** attention. Key algorithmic ideas: **tiling and recomputation**. Upshot: faster training, and better models with longer sequences. Code: github.com/Dao-AILab/flash-attention (in PyTorch, `F.scaled_dot_product_attention` dispatches to it automatically).

---
## 4. State space models (slides credit: Albert Gu)

**Deep sequence models** share one template (normalization → linear → **sequence transformation** → ...), differing in the sequence layer:
* **RNNs:** ✓ natural autoregressive (causal) models; ✗ slow training on accelerators and poor optimization (vanishing gradients). *Sequential.*
* **Attention (Transformers):** ✓ strong performance, parallelizable; ✗ **quadratic-time training, linear-time inference** (in the sequence length). *Dense interactions.*
* **Selective state spaces (Mamba):** ✓ matches Transformers on LM; ✓ parallelizable training + fast inference; ✓ long context, improving up to million-length sequences.

### 4.1 Three views of a state space model
A **state space model** (Kalman 1960) maps a 1-D input signal $x(t)$ to a 1-D output $y(t)$ through an $N$-dimensional latent state $h(t)$:
* **Continuous:** $h'(t) = Ah(t) + Bx(t)$, $\ y(t) = Ch(t) + Dx(t)$. Operates on signals and sequences.
* **Recurrent** (after discretizing with step size $\Delta$): $h_k = \bar Ah_{k-1} + \bar Bx_k$, $\ y_k = \bar Ch_k + \bar Dx_k$. **Efficient autoregressive computation** of the state.
* **Convolutional:** $y = \bar K * x$, with kernel $\bar K = (\bar C\bar B, \bar C\bar A\bar B, \bar C\bar A^2\bar B, \dots)$. **Parallelizable, nearly-linear computation** (via FFT): the output can be computed **without computing the state**. It **generalizes CNNs** (a learned, sequence-long filter). (L16 §6 verified the recurrent = convolutional equivalence.)

**Discretization** with a zero-order hold (input held constant over each step of length $\Delta$) gives exactly
$$\bar A = e^{\Delta A},\qquad \bar B = (\Delta A)^{-1}\big(e^{\Delta A} - I\big)\,\Delta B$$

In [ ]:
r = np.random.default_rng(0)
N_s = 4
A = -np.diag(r.uniform(0.5, 3.0, N_s)) + 0.3 * r.normal(size=(N_s, N_s))         # a stable continuous-time system
B, C = r.normal(size=(N_s, 1)), r.normal(size=(1, N_s))
delta = 0.1
A_bar = expm(delta * A)
B_bar = np.linalg.solve(delta * A, (A_bar - np.eye(N_s))) @ (delta * B)            # zero-order hold
x_seq = r.normal(size=60)

h = np.zeros((N_s, 1)); y_rec = []
for xk in x_seq:                                                                    # discrete recurrence
    h = A_bar @ h + B_bar * xk; y_rec.append((C @ h).item())
h = np.zeros((N_s, 1)); y_ode = []; fine = 2000
for xk in x_seq:                                                                    # integrate the ODE finely, input held constant
    for _ in range(fine):
        h = h + (delta / fine) * (A @ h + B * xk)
    y_ode.append((C @ h).item())
print(f"ZOH-discretized recurrence vs fine ODE integration: max difference {np.max(np.abs(np.array(y_rec) - np.array(y_ode))):.1e}")

### 4.2 Linear time invariance, and its limits
S4-style SSMs are **linear time invariant (LTI)**: the parameters $(\Delta, A, B, C)$ are **constant through time**. "LTI SSM" can refer to any model that is a linear recurrence (e.g., LRU) or a global convolution (e.g., Hyena). LTI is what makes the convolutional view (and its efficiency) possible. It's **great for "continuous" domains (audio, images), but not for text.**

**The tradeoffs of sequence models can be understood through their autoregressive state:**
* **RNNs / SSMs:** state = a **fixed-size vector** (compression). ✓ Efficient: constant-time inference, linear-time training. ✗ Poor performance on information-dense modalities (language).
* **Attention:** state = **a cache of the entire history** (no compression). ✓ Strong performance: models all connections and long-range dependencies. ✗ Inefficient: linear-time inference per token, quadratic-time training.

---
## 5. Selection: Mamba (Gu & Dao 2023)

**Selection mechanism:** keep the same 1-D → 1-D map, but make the **parameters depend on the input**: $\Delta_t, B_t, C_t$ are computed from $x_t$ by linear projections. Now the model can decide, per token, what to **write** into its state and what to **ignore**, the content-based gating that LTI models lack (and that LSTMs have, L6).

**Why LTI models can't do this:** an LTI model's output is a fixed linear function (convolution) of its inputs. Consider a **selective** task: tokens carry a *value* and a *flag*, and the output should be the **sum of the flagged values**, ignoring the rest. That's a sum of products (flag × value), which no fixed linear filter can compute. A selective SSM with $B_t$ = flag$_t$ does it exactly:

In [ ]:
r = np.random.default_rng(0)
T_len, n_seq = 40, 3000
vals = r.normal(size=(n_seq, T_len)); flags = (r.random((n_seq, T_len)) < 0.2).astype(float)
target = (vals * flags).sum(1)                                     # sum of the flagged values

# Best possible LTI model: y = sum_t K1[t] * value_t + K2[t] * flag_t  (any linear filter over both input channels)
Xlti = np.c_[vals, flags]; K_fit = np.linalg.lstsq(Xlti[:2000], target[:2000], rcond=None)[0]
pred_lti = Xlti[2000:] @ K_fit
r2 = lambda p_: 1 - np.mean((p_ - target[2000:]) ** 2) / np.var(target[2000:])

# Selective SSM: h_t = A_t h_{t-1} + B_t x_t, with B_t = flag_t (input-dependent), A_t = 1, x_t = value_t, y = h_T
def selective_ssm(vals, flags):
    h = np.zeros(len(vals))
    for t in range(vals.shape[1]):
        A_t, B_t = 1.0, flags[:, t]                                # selection: write only when flagged
        h = A_t * h + B_t * vals[:, t]
    return h
print(f"best LTI (linear filter) model: R^2 = {r2(pred_lti):.3f}")
print(f"selective SSM (B_t = flag_t):    R^2 = {r2(selective_ssm(vals[2000:], flags[2000:])):.3f}")

The best linear filter explains almost nothing (it can only add up values and flags with fixed weights); the selective model is exact. **But wait: LTI models were necessary for efficiency.** With input-dependent parameters there's no fixed convolution kernel, and naively materializing an expanded state for every position ($B\times L\times D\times N$) in HBM is too slow and too big.

**Hardware-aware state expansion.** Idea: **only materialize the expanded state in more efficient levels of the memory hierarchy.** Load $\Delta, A, B, C$ from HBM to SRAM, run the recurrence (discretize + scan) **in SRAM**, and write only the outputs $y$ back to HBM: the FlashAttention lesson applied to recurrences. The recurrence is computed with a **parallel scan**: $h_t = a_th_{t-1} + b_t$ is a composition of affine maps, and composing affine maps is **associative**, $(a_1, b_1)\circ(a_2, b_2) = (a_1a_2,\ a_2b_1 + b_2)$, so all prefixes can be computed in $O(\log T)$ parallel steps:

In [ ]:
def sequential_scan(a, b):
    h, out = 0.0, []
    for at, bt in zip(a, b):
        h = at * h + bt; out.append(h)
    return np.array(out)

def parallel_scan(a, b):
    # Hillis-Steele inclusive scan with the associative operator (a1,b1) o (a2,b2) = (a1*a2, a2*b1 + b2).
    a, b = a.copy(), b.copy(); steps, shift = 0, 1
    while shift < len(a):
        a_prev = np.r_[np.ones(shift), a[:-shift]]; b_prev = np.r_[np.zeros(shift), b[:-shift]]
        a, b = a_prev * a, a * b_prev + b                        # every position updated in parallel
        shift *= 2; steps += 1
    return b, steps

r = np.random.default_rng(1)
a = r.uniform(0.5, 1.0, 1024); bvec = r.normal(size=1024)        # input-dependent decay a_t and input b_t (selective!)
par, steps = parallel_scan(a, bvec)
print(f"parallel scan == sequential recurrence: {np.allclose(par, sequential_scan(a, bvec))}   "
      f"({steps} parallel steps instead of {len(a)} sequential ones)")

**The Mamba block** simplifies prior SSM architectures (H3) and gated MLPs into one homogeneous block: a linear projection, a short convolution, the selective SSM, and multiplicative gating with a parallel branch, with no attention and no separate MLP block.

---
## 6. Applications and implications

* **Language modeling, scaling laws:** compared with a GPT-3 recipe Transformer, prior SSM-style models (H3, Hyena, RWKV, RetNet), and **Transformer++** (the strong Llama recipe), **Mamba is the first attention-free model to compete with strong modern Transformers** across scales (in the paper's experiments, up to ~1–3B parameters).
* **Zero-shot evaluations:** Mamba matches or beats Transformers of similar size.
* **DNA pretraining:** next-base-pair prediction, where interactions can be extremely long-range; unlike LTI models, Mamba **improves with longer context**.
* **Audio:** improved perplexity up to **1M-length sequences** (~1 minute of audio).

**Summary of Mamba:** matches or beats the strongest Transformer architecture on language; key ideas: the **selection mechanism** and **hardware-aware state expansion**; upshot: better models with **linear (instead of quadratic) scaling** in sequence length. Code: github.com/state-spaces/mamba.

**Implications for foundation models.** LLMs have many mysterious properties and affordances: prompting, in-context learning, instruction tuning, RLHF, cross-modal transfer. There's extensive work (and speculation) on how *statistical modeling* assumptions lead to them. **What if the architecture is the root of these phenomena?** With SSMs: **Scenario 1:** SSMs work as well as Transformers downstream → a deeper understanding of foundation models (the properties come from the data and objective). **Scenario 2:** SSMs are missing some downstream capabilities → we learn which capabilities attention provides. 🔎 Since 2024, evidence has accumulated for a bit of both: pure SSMs lag on tasks needing precise **retrieval/copying** from long contexts (a fixed-size state must compress), and **hybrids** that interleave a few attention layers with many SSM/linear-recurrent layers (e.g., Jamba, and later production models) have become a popular way to get most of the efficiency with Transformer-level recall.

---
## 7. 🔎 Beyond the slides: deployment essentials

The slides cover the kernels; here's what else determines the cost of serving a model.

### 7.1 KV-cache memory
During generation, each token's keys and values are cached for every layer (L8 §8.4). Per sequence: $2\times\text{layers}\times\text{KV heads}\times d_{\text{head}}\times\text{tokens}\times\text{bytes}$. **Grouped-query attention** (GQA; several query heads share one KV head) shrinks it directly:

In [ ]:
def kv_cache_gb(layers, kv_heads, d_head, tokens, batch=1, bytes_=2):
    return 2 * layers * kv_heads * d_head * tokens * batch * bytes_ / 1e9
for name, layers, q_heads, kv_heads, d_head in [("70B, full multi-head", 80, 64, 64, 128), ("70B, GQA (8 KV heads)", 80, 64, 8, 128)]:
    print(f"{name:24s}: KV cache {kv_cache_gb(layers, kv_heads, d_head, 4096):6.2f} GB per 4K-token sequence, "
          f"{kv_cache_gb(layers, kv_heads, d_head, 4096, batch=32):7.1f} GB for a batch of 32")

At long contexts and large batches the KV cache, not the weights, fills GPU memory. Serving systems therefore use GQA, **paged attention** (vLLM: store the cache in fixed-size blocks, like virtual memory pages, so memory isn't wasted on fragmentation), and **continuous batching** (add new requests to the running batch as others finish, instead of waiting for the whole batch).

### 7.2 Quantization
Decoding is memory-bound (each step reads every weight once to produce one token), so **storing weights in fewer bits** speeds it up nearly proportionally. Simple **absmax** quantization maps each value to an integer: $q = \text{round}(w/s)$, $s = \max|w|/127$ for int8. A single **outlier** forces a large scale for its whole group, wasting precision; finer granularity (per output channel, or per group of 64–128 weights) fixes most of that. (Real methods: GPTQ and AWQ for 4-bit weights; LLM.int8() handles activation outliers separately.) In the results below, notice that even groups of 64 leave a sizable int4 error: the outlier channel still inflates the scale of the group it falls in. That's exactly why methods like AWQ (rescale salient channels before quantizing) and LLM.int8() (keep outlier features in higher precision) treat outliers specially.

In [ ]:
r = np.random.default_rng(0)
W = r.normal(0, 0.02, (1024, 1024)); W[:, 7] *= 30                    # one outlier input channel (common in LLMs)
X = r.normal(size=(64, 1024))

def quantize(W, bits, group=None, axis_per_row=False):
    qmax = 2 ** (bits - 1) - 1
    if group:                                                          # groups of `group` weights along each row
        Wg = W.reshape(W.shape[0], -1, group); s = np.abs(Wg).max(-1, keepdims=True) / qmax
        return (np.round(Wg / s).clip(-qmax, qmax) * s).reshape(W.shape)
    s = (np.abs(W).max(1, keepdims=True) if axis_per_row else np.abs(W).max()) / qmax
    return np.round(W / s).clip(-qmax, qmax) * s

ref = X @ W.T
print(f"{'scheme':34s} {'bits/weight':>11s} {'relative output error':>22s}")
for name, Wq, bits in [("int8, one scale for the matrix", quantize(W, 8), 8), ("int8, per-row scales", quantize(W, 8, axis_per_row=True), 8),
                       ("int4, per-row scales", quantize(W, 4, axis_per_row=True), 4), ("int4, groups of 64", quantize(W, 4, group=64), 4)]:
    err = np.linalg.norm(X @ Wq.T - ref) / np.linalg.norm(ref)
    print(f"{name:34s} {bits:11d} {err:22.2%}")
print(f"memory for a 7B model: fp16 {7e9 * 2 / 1e9:.0f} GB, int8 {7e9 / 1e9:.0f} GB, int4 {7e9 * 0.5 / 1e9:.1f} GB (+ small scale overhead)")

### 7.3 Speculative decoding (Leviathan et al. 2023; Chen et al. 2023)
Decoding is memory-bound, so verifying several tokens at once costs about the same as generating one. **Speculative decoding** uses a small **draft** model $q$ to propose $\gamma$ tokens, and the large **target** model $p$ to check them all in one forward pass. Each drafted token $x$ is **accepted** with probability $\min(1, p(x)/q(x))$; at the first rejection, a token is resampled from the residual distribution $\propto\max(0, p - q)$. **This produces samples from exactly $p$**: a pure speedup, no approximation. With per-token acceptance rate $\alpha = \sum_x\min(p(x), q(x))$, each target call yields on average $\frac{1-\alpha^{\gamma+1}}{1-\alpha}$ tokens.

In [ ]:
r = np.random.default_rng(0)
Vocab = 8
p_target = softmax(r.normal(size=Vocab) * 1.5)                         # target model's next-token distribution
q_draft = softmax(np.log(p_target) + r.normal(size=Vocab) * 0.7)        # a similar but imperfect draft model

def speculative_step(r_):
    x = r_.choice(Vocab, p=q_draft)                                     # draft proposes
    if r_.random() < min(1.0, p_target[x] / q_draft[x]):                 # target accepts with prob min(1, p/q)
        return x, True
    resid = np.maximum(p_target - q_draft, 0); resid /= resid.sum()       # on rejection: resample from the residual
    return r_.choice(Vocab, p=resid), False

samples, accepted = zip(*[speculative_step(r) for _ in range(200_000)])
emp = np.bincount(samples, minlength=Vocab) / len(samples)
alpha = np.minimum(p_target, q_draft).sum()
print(f"total variation distance between speculative samples and the target distribution: {0.5 * np.abs(emp - p_target).sum():.4f}")
print(f"acceptance rate: measured {np.mean(accepted):.3f}, theory alpha = sum min(p, q) = {alpha:.3f}")
for gamma in [1, 3, 5, 8]:
    print(f"  draft length {gamma}: expected tokens per target forward pass = {(1 - alpha ** (gamma + 1)) / (1 - alpha):.2f}")

The samples match the target distribution (the tiny TV distance is sampling noise), and with a good draft model each expensive forward pass of the big model yields several tokens. Real speedups are typically 2–3×, depending on how often the draft agrees with the target.

---
## 8. Summary

| Topic | Takeaway |
|---|---|
| The bottleneck | Attention is bound by **memory traffic** (writing/reading $N\times N$ matrices in HBM), not FLOPs |
| Roofline | Arithmetic intensity vs. the ridge point (~200 FLOPs/byte on an A100); elementwise ops are memory-bound |
| FlashAttention | **Tiling** + online-softmax **rescaling** (exact), **recomputation** in the backward pass from the stored log-sum-exp; 2–4× faster, 10–20× less memory, longer context |
| FA-2 / H100 / Flash-Decoding | Fewer non-matmul FLOPs, sequence-parallelism, new hardware features; split the KV cache for decoding + a log-sum-exp reduction |
| SSM views | Continuous ↔ recurrent (ZOH: $\bar A = e^{\Delta A}$) ↔ convolutional |
| State tradeoff | Fixed-size state (efficient, compresses) vs. full cache (powerful, expensive) |
| Selection (Mamba) | Input-dependent $\Delta, B, C$; computed by a hardware-aware **parallel scan** in SRAM; linear scaling, Transformer-level LM quality |
| 🔎 Serving | KV cache (GQA, paged attention, continuous batching), **quantization** (per-channel/group scales), **speculative decoding** (exact) |

### Exercises
1. Add **causal masking** to `flash_attention_forward` (skip key blocks entirely above the diagonal) and count how much traffic and compute that saves.
2. Measure HBM traffic of the simulated FlashAttention for block sizes 16, 32, 64, 128 and plot it against $1/B_r$.
3. Implement **online softmax** for a single long vector streamed in chunks (the 1-D version of §2.1), and use it to compute log-sum-exp in one pass.
4. Make the selective SSM in §5 *learn* $B_t = w^\top x_t$ by gradient descent on the selective-sum task, starting from $w = 0$.
5. In §7.3, make the draft model worse (larger noise) and plot the acceptance rate and expected speedup.

### References
* Dao, Fu, Ermon, Rudra & Ré 2022, *FlashAttention*; Dao 2023, *FlashAttention-2*; Shah et al. 2024 (FlashAttention-3); Dao, Haziza, Massa & Sizov 2023 (Flash-Decoding).
* Gu, Goel & Ré 2022 (S4); Gu & Dao 2023, *Mamba*; Kalman 1960; Blelloch 1990 (prefix sums).
* Tay et al. 2021 (Long Range Arena); Williams, Waterman & Patterson 2009 (the roofline model); He 2022, *Making Deep Learning Go Brrrr*.
* Kwon et al. 2023 (vLLM, PagedAttention); Ainslie et al. 2023 (GQA); Dettmers et al. 2022 (LLM.int8()); Frantar et al. 2023 (GPTQ); Lin et al. 2024 (AWQ).
* Leviathan, Kalman & Matias 2023; Chen et al. 2023 (speculative sampling).